# Lab 1: Song Similarity with Audio-Feature Vectors

In the previous notebook we used pretrained word embeddings to explore dot products, cosine
similarity, and angles between vectors. In this lab you'll apply the exact same tools to a very
different kind of vector: **songs**, represented by their Spotify audio features (danceability,
energy, loudness, tempo, etc.).

Unlike a user-item ratings matrix (e.g. MovieLens, Amazon reviews), this dataset has essentially
**no missing data** — Spotify computes these audio features for every track, so every song is a
complete vector, no gaps to work around.

Dataset: [`spotify_songs.csv`](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-01-21/readme.md)
(TidyTuesday, Jan 2020), ~30,000 tracks pulled from Spotify playlists.

## Part 0 sets things up for you (data loading, cleaning, helper functions) — just run those cells.
## Part 1 is yours: answer the questions using the tools from Part 0.

## Part 0: Setup (provided)

Run the cells below as-is.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

### Load the data

Downloads the CSV the first time (and caches it in `../data/`, so later runs don't need the
network).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
DATA_PATH = Path("../data/spotify_songs.csv")

DATA_PATH.parent.mkdir(exist_ok=True)
if not DATA_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

songs = pd.read_csv(DATA_PATH)
print(f"loaded {len(songs)} rows")
songs.head()

### Clean the data

A couple of things need fixing before we can treat each row as one vector per song:

1. The same track shows up multiple times if it appears on multiple playlists (with identical
   audio features) — we only want one row per unique `track_id`.
2. A handful of rows are missing the track/artist name, which we need for displaying results.

In [ ]:
songs = songs.dropna(subset=["track_name", "track_artist"])
songs = songs.drop_duplicates(subset="track_id", keep="first")
songs = songs.reset_index(drop=True)

print(f"{len(songs)} unique songs remain")

### Build the feature matrix

We represent each song as a vector of its numeric audio features. Two things need handling:

- `key` (musical key, 0-11) and `mode` (major/minor, 0/1) are categorical labels, not
  continuous magnitudes, so we leave them out of the similarity vector.
- The remaining features are on very different scales — `duration_ms` is in the hundreds of
  thousands, `tempo` is in the tens/hundreds, while `danceability`, `energy`, etc. range from
  0 to 1. A raw dot product would be dominated by whichever feature happens to have the largest
  numbers, which has nothing to do with musical similarity. We fix this by **standardizing**
  each feature (subtract its mean, divide by its standard deviation) so every feature
  contributes on a comparable scale.

In [ ]:
feature_columns = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_ms",
]

X = songs[feature_columns].to_numpy()
X = (X - X.mean(axis=0)) / X.std(axis=0)

print(f"feature matrix shape: {X.shape}")
# X[i] is now the vector for songs.loc[i]

### Helper functions (provided)

A few small helpers for looking up songs and getting a link to listen on Spotify — each one is
followed by an example call showing what it does. (You'll write the dot-product /
cosine-similarity / angle functions yourself in Part 1.)

In [ ]:
def find_song(name_contains, artist_contains=None):
    """Look up songs by a (partial, case-insensitive) name and optional artist."""
    matches = songs[songs["track_name"].str.contains(name_contains, case=False)]
    if artist_contains is not None:
        matches = matches[matches["track_artist"].str.contains(artist_contains, case=False)]
    return matches[["track_name", "track_artist", "track_id"]]

In [ ]:
matches = find_song("shape of you", "ed sheeran")
matches

In [ ]:
def spotify_url(track_id):
    return f"https://open.spotify.com/track/{track_id}"

In [ ]:
spotify_url(matches.iloc[0]["track_id"])

In [ ]:
def describe_song(index):
    row = songs.loc[index]
    return f"{row['track_name']} — {row['track_artist']} ({spotify_url(row['track_id'])})"

In [ ]:
describe_song(matches.index[0])

### Pick a target song

`find_song` can return more than one match (covers, remixes, re-releases all have their own
`track_id`) — inspect the results and pick the row you actually want.

In [ ]:
find_song("bad guy", "billie eilish")

In [ ]:
# there are a couple of matches above (the original and the Bieber remix) — we want the original
target_index = find_song("bad guy", "billie eilish")
target_index = target_index[target_index["track_name"] == "bad guy"].index[0]

print(describe_song(target_index))

## Part 1: Exercises

You'll need your own `dot`, `norm`, `cosine_similarity`, and `angle_degrees` functions here
(same definitions as Lab 0) — write those first, then use them together with `X`, `songs`,
`target_index`, and the helper functions above to answer the questions below.

Feel free to change `target_index` to a song of your choice (use `find_song` to look up its
index) before or while answering these.

In [ ]:
# TODO: define dot, norm, cosine_similarity, angle_degrees


### Question 1

Find the song **most similar** to `target_index` (excluding the target song itself). Print its
name, artist, its Spotify link (`describe_song` gives you all three), and the angle (in degrees)
between it and the target song.

In [ ]:
# TODO: find the most similar song to `target_index` and print its name/artist/link and angle


### Question 2

Find the song **least similar** to `target_index` (again excluding the target song itself).
Print the same information: name, artist, Spotify link, and angle.

In [ ]:
# TODO: find the least similar song to `target_index` and print its name/artist/link and angle


### Question 3

Listen to (or think about) the target song, the most similar song, and the least similar song.
**Do the results make sense?** Justify your answer in terms of the actual audio features
involved — which features are close together / far apart between the songs?

_Your answer here._

### More questions

_(to be added)_